# 第三章 3.6 技术验证\n本 Notebook 只汇总验证，不重新估计正式模型。正式推断来自 Stata/MP 18。

In [ ]:
from pathlib import Path

import pandas as pd

from src.chapter3_stata_crosscheck import compare_point_estimates, validate_wcb_coherence

root = Path('results/chapter3')
baseline = pd.read_csv(root / 'stata_baseline_inference.csv')
timing = pd.read_csv(root / 'stata_timing_robustness.csv')
measurement = pd.read_csv(root / 'stata_measurement_robustness.csv')
leaveout = pd.read_csv(root / 'stata_leave_one_province_out.csv')
weights = pd.read_csv(root / 'stata_weight_sensitivity.csv')


In [ ]:
panel = pd.read_parquet('data/processed/research_panel_policy.parquet')
lagged = pd.read_parquet('data/processed/research_panel_policy.parquet')
assert len(panel) == 240
assert baseline.province_clusters.eq(7).all()
assert timing.loc[timing.model.eq('lagged_primary'), 'N'].le(200).all()
display(timing.loc[timing.model.eq('lagged_primary'), ['outcome','N']])


In [ ]:
python = pd.read_csv(root / 'python_baseline_point_estimates.csv')
crosscheck = compare_point_estimates(python, baseline, tolerance=1e-7)
assert crosscheck.passed.all()
display(crosscheck)


In [ ]:
assert baseline.wcb_weight.eq('Webb').all()
assert baseline.wcb_reps.eq(10000).all()
assert baseline.wcb_seed.eq(20260918).all()
assert baseline.wcb_ptype.eq('equal-tailed').all()
assert validate_wcb_coherence(baseline).coherent.all()
assert validate_wcb_coherence(timing).coherent.all()
assert validate_wcb_coherence(measurement).coherent.all()
display(weights[['weight','wcb_p','wcb_ci_low','wcb_ci_high']])


In [ ]:
old = pd.read_csv(root / 'pre_3_6' / 'baseline_inference.csv')
display(old[['model','beta','wcb_p']])
display(baseline[['model','beta','wcb_p']])
assert (root / 'pre_3_6').exists()
